In [1]:
import json
import pandas as pd



In [2]:
def extract_products(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)

    df = pd.DataFrame(data)
    return df
extract_df = extract_products("messy_products_153.json")

In [3]:
def data_inspection(df):

    print(df.head())

    df.info()

    print(df.describe())

    print("Duplicates:", df.astype(str).duplicated().sum())

    print("Missing values:")
    print(df.isnull().sum())
    return df
inspected_df = data_inspection(extract_df )

   id       sku             product_name        category   price currency  \
0   1  SKU-1001           Wireless Mouse          Office  133.77      USD   
1   2  SKU-1002        Ultra-Light Mouse     Electronics  121.04      USD   
2   3  SKU-1003  Heavy-Duty Water Bottle  Home & Kitchen  232.87      USD   
3   4  SKU-1004         Compact Backpack  Home & Kitchen  119.43      USD   
4   5  SKU-1005         Compact Notebook          Office  406.46      USD   

  stock rating in_stock      supplier created_date             tags  
0    35    4.3     True     OfficeHub   2026-06-13        [popular]  
1   129    3.9     True     OfficeHub   2026-04-19            [eco]  
2    71    4.5     True  ABC Supplies   2026-06-16            [eco]  
3   195    3.2     True  ABC Supplies   2026-04-19   [sale, budget]  
4   186    3.5     True  ABC Supplies   2026-08-05  [sale, popular]  
<class 'pandas.DataFrame'>
RangeIndex: 153 entries, 0 to 152
Data columns (total 12 columns):
 #   Column        Non-

In [4]:
print(inspected_df.dtypes)

id               int64
sku                str
product_name       str
category           str
price           object
currency           str
stock           object
rating          object
in_stock        object
supplier           str
created_date       str
tags            object
dtype: object


In [5]:
cols = ['price', 'stock', 'rating']

inspected_df [cols] = inspected_df [cols].apply(pd.to_numeric, errors='coerce')

In [6]:
print(inspected_df.dtypes)

id                int64
sku                 str
product_name        str
category            str
price           float64
currency            str
stock           float64
rating          float64
in_stock         object
supplier            str
created_date        str
tags             object
dtype: object


In [7]:
def handle_missing_values(df):

    numerical_cols = ['price', 'stock', 'rating']

    for col in numerical_cols:
        df[col] = df[col].fillna(df[col].median())

    categorical_cols = ["sku", "product_name", "category", "currency", "supplier"]

    for col in categorical_cols:
        df[col] = df[col].fillna('unknown')

    df['created_date'] = df['created_date'].fillna('unknown')

    df['tags'] = df['tags'].apply(
        lambda x: [] if x is None or (isinstance(x, float) and pd.isna(x)) else x
    )

    return df
handled_df = handle_missing_values(inspected_df)

In [8]:
print(handled_df.isnull().sum())

id              0
sku             0
product_name    0
category        0
price           0
currency        0
stock           0
rating          0
in_stock        0
supplier        0
created_date    0
tags            0
dtype: int64


In [9]:
nulled_Df = handled_df[handled_df['product_name'].isna()]
print(nulled_Df)


Empty DataFrame
Columns: [id, sku, product_name, category, price, currency, stock, rating, in_stock, supplier, created_date, tags]
Index: []


In [10]:
handled_df[handled_df['supplier'].isna()]

,id,sku,product_name,category,price,currency,stock,rating,in_stock,supplier,created_date,tags


In [11]:
def handle_duplicates(df):

    subset = [col for col in df.columns if col != 'tags']

    df = df.drop_duplicates(subset=subset)

    return df
no_duplicated_df = handle_duplicates(handled_df)
subset = [col for col in no_duplicated_df.columns if col != 'tags']
print(no_duplicated_df.duplicated(subset).sum())


0


In [12]:
def standardize_text(df):

    text_cols = ["sku","product_name", "category","currency","supplier"]

    for col in text_cols:
        df[col] = df[col].astype(str).str.strip().str.lower()

    return df
standardized_df = standardize_text(no_duplicated_df )
print(standardized_df)

      id       sku             product_name        category    price currency  \
0      1  sku-1001           wireless mouse          office  133.770      usd   
1      2  sku-1002        ultra-light mouse     electronics  121.040      usd   
2      3  sku-1003  heavy-duty water bottle  home & kitchen  232.870      usd   
3      4  sku-1004         compact backpack  home & kitchen  119.430      usd   
4      5  sku-1005         compact notebook          office  406.460      usd   
..   ...       ...                      ...             ...      ...      ...   
145  146  sku-1146             smart webcam         outdoor  246.135      usd   
146  147  sku-1147           wireless mouse         outdoor   70.200      usd   
147  148  sku-1148   ultra-light headphones     accessories  492.460      usd   
148  149  sku-1149           wireless mouse          office   73.190      usd   
149  150  sku-1150   portable monitor stand         outdoor  199.990      usd   

     stock  rating in_stock

In [13]:
print(standardized_df.head())
print(standardized_df.tail())

   id       sku             product_name        category   price currency  \
0   1  sku-1001           wireless mouse          office  133.77      usd   
1   2  sku-1002        ultra-light mouse     electronics  121.04      usd   
2   3  sku-1003  heavy-duty water bottle  home & kitchen  232.87      usd   
3   4  sku-1004         compact backpack  home & kitchen  119.43      usd   
4   5  sku-1005         compact notebook          office  406.46      usd   

   stock  rating in_stock      supplier created_date             tags  
0   35.0     4.3     True     officehub   2026-06-13        [popular]  
1  129.0     3.9     True     officehub   2026-04-19            [eco]  
2   71.0     4.5     True  abc supplies   2026-06-16            [eco]  
3  195.0     3.2     True  abc supplies   2026-04-19   [sale, budget]  
4  186.0     3.5     True  abc supplies   2026-08-05  [sale, popular]  
      id       sku            product_name     category    price currency  \
145  146  sku-1146          

In [14]:
def filter_invalid_records(df):

    before = len(df)

    df = df[df['id'].notna()]
    df = df[df['price'] >= 0]
    df = df[df['stock'] >= 0]
    df = df[df['rating'].between(0, 5)]

    after = len(df)

    print("Invalid records removed:", before - after)
    print(df.isnull().sum())

    return df
validated_df = filter_invalid_records(standardized_df)
print(validated_df.isnull().sum())

Invalid records removed: 6
id              0
sku             0
product_name    0
category        0
price           0
currency        0
stock           0
rating          0
in_stock        0
supplier        0
created_date    0
tags            0
dtype: int64
id              0
sku             0
product_name    0
category        0
price           0
currency        0
stock           0
rating          0
in_stock        0
supplier        0
created_date    0
tags            0
dtype: int64


In [15]:
validated_df.to_csv("transformed_products.csv")

In [16]:
df = pd.read_csv("transformed_products.csv")
print(df.head())

print(df.info())

print(df.describe())

print("Duplicates:", df.astype(str).duplicated().sum())

print("Missing values:")
print(df.isnull().sum())

   Unnamed: 0  id       sku             product_name        category   price  \
0           0   1  sku-1001           wireless mouse          office  133.77   
1           1   2  sku-1002        ultra-light mouse     electronics  121.04   
2           2   3  sku-1003  heavy-duty water bottle  home & kitchen  232.87   
3           3   4  sku-1004         compact backpack  home & kitchen  119.43   
4           4   5  sku-1005         compact notebook          office  406.46   

  currency  stock  rating in_stock      supplier created_date  \
0      usd   35.0     4.3     True     officehub   2026-06-13   
1      usd  129.0     3.9     True     officehub   2026-04-19   
2      usd   71.0     4.5     True  abc supplies   2026-06-16   
3      usd  195.0     3.2     True  abc supplies   2026-04-19   
4      usd  186.0     3.5     True  abc supplies   2026-08-05   

                  tags  
0          ['popular']  
1              ['eco']  
2              ['eco']  
3   ['sale', 'budget']  
4  

In [17]:
df[df['product_name'].isna()]


,Unnamed: 0,id,sku,product_name,category,price,currency,stock,rating,in_stock,supplier,created_date,tags
115,120,121,sku-1121,NaN,accessories,484.13,usd,131.0,3.8,True,abc supplies,2026-07-13,"['sale', 'budget']"


In [18]:
df[df['supplier'].isna()]

,Unnamed: 0,id,sku,product_name,category,price,currency,stock,rating,in_stock,supplier,created_date,tags
27,27,28,sku-1028,ultra-light water bottle,office,246.135,usd,200.0,3.5,True,NaN,15/01/2026,"['eco', 'popular', 'new']"
89,93,94,sku-1094,wireless mouse,outdoor,330.980,usd,234.0,3.1,True,NaN,01/15/2026,sale
